# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tejupriyakukkala-creator/flyrank-task1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Content Refresh & Decay Prediction  
**Goal:** Build a practical, non-production Content Action Playbook that blends machine learning probability scores with transparent rule safeguards to categorize content into actionable refresh workflows, define operational boundaries, establish human review checklists and no-go rules, set monitoring drift triggers, and export receipts and figures for research documentation.

## 1. Ranked actions + reason codes

### Playbook Scoring & Action Categorization
The playbook builds a **Blended Action Score** combining calibrated model decay probabilities ($70\%$ weight) with transparent rule risk sub-scores ($30\%$ weight):

$$\text{action\_score} = 0.70 \times \text{prob\_model} + 0.30 \times \text{score\_rule}$$

Each flagged item is assigned a human-understandable **Reason Code** and mapped to a specific **Action Category**:

| Action Category | Reason Code Trigger | Operational Workflow | Expected Impact |
|---|---|---|---|
| `expand_and_refresh` | `thin_visible_page` (`word_count < 1200` & `impressions_90d >= 250`) | Expand content depth, add sub-headers, and flesh out thin sections. | High ($+$ depth relevance) |
| `refresh_and_review_ctr` | `low_ctr_visible_page` (`impressions_90d >= 500` & `ctr < 0.5%`) | Rewrite title tags, meta descriptions, and featured snippet hooks. | Medium ($+$ CTR optimization) |
| `refresh` | `stale_visible_page` (stale $\ge 180$d) OR `striking_decay_risk` (pos 4-20) | Update facts, statistics, outbound links, and internal link equity. | High ($+$ SERP recovery) |
| `monitor` | Default fallback | Maintain baseline tracking without immediate editorial intervention. | Low (Baseline monitoring) |

In [1]:
import pandas as pd
import numpy as np
import pathlib
import urllib.request
import json
from sklearn.ensemble import HistGradientBoostingClassifier

# Load starter dataset (robust for local workspace OR Google Colab direct execution)
data_path = pathlib.Path('data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = pathlib.Path('../../data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = pathlib.Path('../data/raw/content_refresh_anonymized.csv')

if not data_path.exists():
    print("Local dataset not found. Fetching raw dataset from GitHub for Colab...")
    raw_url = "https://raw.githubusercontent.com/tejupriyakukkala-creator/flyrank-task1/main/data/raw/content_refresh_anonymized.csv"
    data_dir = pathlib.Path('data/raw')
    data_dir.mkdir(parents=True, exist_ok=True)
    data_path = data_dir / 'content_refresh_anonymized.csv'
    urllib.request.urlretrieve(raw_url, data_path)
    print(f"Successfully downloaded raw dataset to {data_path.as_posix()}")

df = pd.read_csv(data_path)

numeric_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr',
    'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['is_declining_label'] = (df['trend_direction'].fillna('').str.lower() == 'down').astype(int)

# Fit ensemble model for probability predictions
RANDOM_STATE = 42
model = HistGradientBoostingClassifier(max_depth=6, max_iter=100, random_state=RANDOM_STATE)
model.fit(df[numeric_cols], df['is_declining_label'])
df['model_prob'] = model.predict_proba(df[numeric_cols])[:, 1]

# Compute transparent rule sub-score
vis = df['impressions_90d'].rank(pct=True).fillna(0)
fresh = df['days_since_last_update'].rank(pct=True).fillna(0)
strik = np.where((df['avg_position'] >= 4) & (df['avg_position'] <= 25), 1.0, np.where(df['avg_position'] > 25, 0.5, 0.2))
stale_flag = np.where((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500), 1.0, 0.0)
df['rule_score'] = (0.35 * vis + 0.30 * fresh + 0.20 * strik + 0.15 * stale_flag).clip(0, 1)

# Blended Playbook Action Score
df['final_action_score'] = 0.70 * df['model_prob'] + 0.30 * df['rule_score']

def get_reasons(row):
    r = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        r.append('stale_visible_page')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        r.append('low_ctr_visible_page')
    if 4 <= row['avg_position'] <= 20 and row['days_since_last_update'] >= 90:
        r.append('striking_decay_risk')
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        r.append('thin_visible_page')
    if not r:
        r.append('general_refresh_review')
    return '|'.join(r)

def get_action(row):
    reasons = set(str(row['reason_codes']).split('|'))
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons or 'striking_decay_risk' in reasons:
        return 'refresh'
    return 'monitor'

df['reason_codes'] = df.apply(get_reasons, axis=1)
df['suggested_action'] = df.apply(get_action, axis=1)
df['playbook_rank'] = df['final_action_score'].rank(method='first', ascending=False).astype(int)

df_sorted = df.sort_values('playbook_rank').reset_index(drop=True)

print('=== PLAYBOOK ACTION SUMMARY (30,000 PAGES) ===')
print(df_sorted['suggested_action'].value_counts().to_string())

print('\n=== TOP 5 PLAYBOOK QUEUE PREVIEW ===')
preview_cols = ['playbook_rank', 'content_id', 'client_id', 'final_action_score', 'suggested_action', 'reason_codes', 'impressions_90d', 'avg_position', 'days_since_last_update']
print(df_sorted[preview_cols].head(5).to_string(index=False))

=== PLAYBOOK ACTION SUMMARY (30,000 PAGES) ===
suggested_action
monitor                   17876
refresh_and_review_ctr     9741
refresh                    2301
expand_and_refresh           82

=== TOP 5 PLAYBOOK QUEUE PREVIEW ===
 playbook_rank           content_id         client_id  final_action_score       suggested_action                                                reason_codes  impressions_90d  avg_position  days_since_last_update
             1 content_7368877ea310 client_7f2253d7e2            0.917326                refresh                                          stale_visible_page            59472          24.8                     194
             2 content_1bfaa38ff26c client_7f2253d7e2            0.913365                refresh                                          stale_visible_page            25715          22.2                     194
             3 content_482aff19e9cc client_7f2253d7e2            0.908257                refresh                                      

## 2. Intended use and limits

### Intended Operational Use
- **Target Audience:** Content strategists, SEO managers, and editorial teams.
- **Primary Use Case:** Weekly decision-support tool to prioritize human editorial capacity toward high-probability organic decay candidates.
- **Portfolio Scope:** Designed for multi-site content portfolios with indexable search history.

### Explicit Operational Boundaries
1. **Non-Production & Non-Automated:** The playbook outputs a decision-support queue; it does **not** auto-edit or auto-publish content updates.
2. **No Ranking Guarantees:** High rank indicates observed decay risk; it does not guarantee immediate SERP recovery upon updating copy.
3. **Cold-Start Restriction:** Content less than 90 days old (`content_age_days < 90`) is out of scope due to lack of historical search performance signal.

## 3. Human review + the no-go list

### Mandatory Human Review Checklist (3 Gates)
Before acting on any flagged page in the queue, an editor must verify:
1. **Search Intent Alignment:** Check if SERP layout shifted toward non-text elements (e.g. calculators, video snippets, or sponsored ad blocks).
2. **Internal Cannibalization Check:** Verify no newer post on the same domain targets the same head keyword.
3. **Conversion Protection:** Confirm the page is not an active high-converting landing page where copy rewrites could degrade lead generation.

### The No-Go List (Never Automate / Exclude from Bulk Refresh)
- **Core Product & Conversion Pages:** Core pricing, signup, or brand landing pages.
- **Regulated Content (YMYL):** Legal terms, privacy policies, medical advice, or financial compliance documentation.
- **Active Site Migration Pages:** Content undergoing URL redirect mapping or domain re-platforming.

## 4. Monitoring / retrain triggers

### Model & Metric Drift Signals
To ensure recommendations remain reliable over time, the system monitors:
- **Precision Drift:** Monthly audit sampling of top-50 flagged items. If audited Precision@50 drops below **70.0%**, trigger model retraining.
- **Base Rate Shift:** If the baseline decline rate across client portfolios shifts by >15% following search engine core algorithm updates.
- **Feature Distribution Drift:** Detect significant changes in missingness or GSC position tracking metrics.

## 5. Exports for the paper

### Exporting Receipts, Metrics, and Figures
We generate and export all required receipts for research paper traceability:
- Queue CSV: `work/outputs/final_action_queue.csv` (gitignored)
- Metrics JSON: `work/outputs/playbook_summary_metrics.json` (committed)
- Reusable Figures: `work/figures/action_distribution.png` and `work/figures/precision_at_k_comparison.png` (committed)

In [2]:
import matplotlib.pyplot as plt

# Ensure output directories
output_dir = pathlib.Path('work/outputs')
fig_dir = pathlib.Path('work/figures')
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Export Final Action Queue CSV
queue_cols = [
    'content_id', 'client_id', 'playbook_rank', 'final_action_score',
    'model_prob', 'rule_score', 'suggested_action', 'reason_codes',
    'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'days_since_last_update', 'word_count'
]
csv_queue_path = output_dir / 'final_action_queue.csv'
df_sorted[queue_cols].to_csv(csv_queue_path, index=False)
print(f'Wrote final action queue CSV to: {csv_queue_path.as_posix()}')

# 2. Export Summary Metrics JSON
metrics_payload = {
    'total_content_items': len(df_sorted),
    'base_rate_declining': round(float(df['is_declining_label'].mean()), 4),
    'action_distribution': df_sorted['suggested_action'].value_counts().to_dict(),
    'precision_metrics': {
        'precision_at_20': round(float(df_sorted.head(20)['is_declining_label'].mean()), 4),
        'precision_at_50': round(float(df_sorted.head(50)['is_declining_label'].mean()), 4),
        'precision_at_100': round(float(df_sorted.head(100)['is_declining_label'].mean()), 4),
        'precision_at_500': round(float(df_sorted.head(500)['is_declining_label'].mean()), 4)
    },
    'scoring_weights': {
        'model_prob_weight': 0.70,
        'rule_score_weight': 0.30
    }
}
json_metrics_path = output_dir / 'playbook_summary_metrics.json'
with open(json_metrics_path, 'w') as f:
    json.dump(metrics_payload, f, indent=2)
print(f'Wrote playbook metrics JSON to: {json_metrics_path.as_posix()}')

# 3. Export Reusable Figures
# Figure A: Action Distribution Chart
plt.figure(figsize=(8, 5))
action_counts = df_sorted['suggested_action'].value_counts()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
action_counts.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Recommended Playbook Action Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Action Type', fontsize=12)
plt.ylabel('Number of Content Items', fontsize=12)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
fig1_path = fig_dir / 'action_distribution.png'
plt.savefig(fig1_path, dpi=300)
plt.close()
print(f'Saved figure to: {fig1_path.as_posix()}')

# Figure B: Precision@K Comparison Chart
k_vals = [10, 20, 50, 100, 200, 500]
rule_p_k = [float(df.sort_values('rule_score', ascending=False).head(k)['is_declining_label'].mean() * 100) for k in k_vals]
model_p_k = [float(df_sorted.head(k)['is_declining_label'].mean() * 100) for k in k_vals]
base_rate_pct = float(df['is_declining_label'].mean() * 100)

plt.figure(figsize=(8, 5))
plt.plot(k_vals, model_p_k, marker='o', linewidth=2.5, label='Blended Playbook Score (Model + Rule)', color='#1f77b4')
plt.plot(k_vals, rule_p_k, marker='s', linewidth=2.0, linestyle='--', label='Rule Baseline Only', color='#ff7f0e')
plt.axhline(base_rate_pct, color='red', linestyle=':', label=f'Base Rate ({base_rate_pct:.1f}%)')
plt.title('Precision@K: Blended Playbook vs Rule Baseline', fontsize=14, fontweight='bold')
plt.xlabel('Top K Content Items Ranked', fontsize=12)
plt.ylabel('Precision@K (%)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
fig2_path = fig_dir / 'precision_at_k_comparison.png'
plt.savefig(fig2_path, dpi=300)
plt.close()
print(f'Saved figure to: {fig2_path.as_posix()}')

Wrote final action queue CSV to: work/outputs/final_action_queue.csv
Wrote playbook metrics JSON to: work/outputs/playbook_summary_metrics.json
Saved figure to: work/figures/action_distribution.png
Saved figure to: work/figures/precision_at_k_comparison.png


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere (all IDs pseudonymized)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.